# EIGSEP NSIDE=8 Multifrequency Signal Recovery (v000)

Ground-based analog of the BLOOM-21cm multifrequency analysis (`bloom21cm/test_multifreq.py`),
adapted for the EIGSEP experiment at Marjum Pass, Utah.

**Key differences from BLOOM:**
- Observer: `EarthSurface` (fixed site) instead of `LunarOrbit`
- Masking: terrain horizon (`HorizonTerrain.mask`) instead of lunar occultation
- Beam: file-based bowtie (`eigsep_bowtie_v000.npz`) instead of analytic thin-dipole
- Diversity: az/alt drives + Earth rotation instead of orbital tumbling
- Design matrix columns: `[sky pixels | T_gnd | T_rx]` (2 extra params vs BLOOM's 4)

**Key lessons from conditioning diagnostics (`eigsep_fg_diag.py`):**
- 2-hour time sampling causes ~108 pixels to cross the horizon simultaneously →
  design matrix columns are nearly collinear → condition number ~10⁴⁷ → 18000% FG leakage
- 30-minute sampling reduces simultaneous transitions to ~27 pixels, saturating the
  horizon-occultation angular discrimination at NSIDE=8 (~7° pixels, ~28-min transit)
- Never-visible pixels (~95, south/Galactic-center direction) get zero weight automatically;
  always-visible NCP pixels (~20) are individually degenerate but their group contribution
  to the monopole is well-constrained
- Column-norm weighted monopole + eigenmode filter built from the column-norm weighted GSM
  gives ~5% FG/signal; 7 days of Earth rotation is needed for adequate sky coverage

**Analysis flow:**
1. Precompute terrain masks and beam values for all (time, pointing) pairs
2. For each frequency: build design matrix A, simulate observations, solve for sky
3. Extract sky monopole via column-norm weights; apply matched GSM eigenmode filter
4. Chi² comparison against 21cm model library

Assumes perfect beam knowledge (no beam calibration errors).

In [ ]:
import os
import numpy as np
import healpy
import matplotlib.pyplot as plt
from astropy.time import Time
import astropy.units as u

from eigsep_sim import EarthSurface
from eigsep_sim import Beam
from eigsep_sim import HorizonTerrain
from eigsep_sim import Sky
from eigsep_sim.models import T21cmModel
from eigsep_sim.recovery import normal_solve
from eigsep_sim.spectral import gsm_eigenmodes, eigenmode_filter

In [ ]:
# ── Sky / inversion resolution ─────────────────────────────────────────────
NSIDE      = 8
NPIX       = healpy.nside2npix(NSIDE)
PIXEL_AREA = 4 * np.pi / NPIX   # sr

# ── EIGSEP site — Marjum Pass, Utah ───────────────────────────────────────
LAT_DEG, LON_DEG = 39.2, -113.4

# ── Science frequency band (mirrors BLOOM science band) ───────────────────
FREQS_MHZ   = np.linspace(55.0, 150.0, 30)
FREQS_HZ    = FREQS_MHZ * 1e6
N_FREQ      = len(FREQS_MHZ)
DELTA_NU_HZ = float(np.diff(FREQS_MHZ).mean()) * 1e6   # channel width [Hz]

# ── Observation schedule ───────────────────────────────────────────────────
OBS_EPOCH       = Time("2025-01-01")
N_DAYS          = 7
N_TIMES_PER_DAY = 48        # sample every 30 min — saturates horizon-transit discrimination at NSIDE=8
N_TIMES         = N_DAYS * N_TIMES_PER_DAY   # 84

# ── Az/alt pointing grid (drive positions) ─────────────────────────────────
N_AZ    = 8
N_ALT   = 4
AZ_DEG  = np.linspace(0, 360, N_AZ, endpoint=False)
ALT_DEG = np.linspace(15, 75, N_ALT)
N_ORIENT = N_AZ * N_ALT
N_ROWS   = N_TIMES * N_ORIENT

# ── Physical / instrument parameters ──────────────────────────────────────
T_GND_K       = 300.0   # physical temperature of terrain-blocked sky [K]
T_RX_K        = 100.0   # receiver noise temperature [K]
INJ_MODEL_IDX = 0       # T21cmModel index to inject
N_GSM_MODES   = 4       # GSM eigenmodes for foreground filter

print(f"NSIDE={NSIDE}  NPIX={NPIX}")
print(f"Freqs: {FREQS_MHZ[0]:.1f}–{FREQS_MHZ[-1]:.1f} MHz  "
      f"({N_FREQ} channels, Δν={DELTA_NU_HZ/1e6:.2f} MHz)")
print(f"Schedule: {N_TIMES} times × {N_ORIENT} pointings = {N_ROWS} rows")
print(f"Design matrix: ({N_ROWS}, {NPIX+2})  overdetermination ratio {N_ROWS/(NPIX+2):.1f}×")


In [ ]:
# ── Observer, beam, and terrain ────────────────────────────────────────────
obs     = EarthSurface(lat=LAT_DEG, lon=LON_DEG)
beam    = Beam(FREQS_HZ)
terrain = HorizonTerrain.from_packaged_model()

BEAM_NSIDE = healpy.npix2nside(beam.map.shape[0])
print(f"Beam NSIDE: {BEAM_NSIDE}  shape: {beam.map.shape}")

# ── GSM sky model at NSIDE=8 ───────────────────────────────────────────────
print("Loading GSM at NSIDE=8 …", flush=True)
sky = Sky.from_gsm(NSIDE, FREQS_HZ, n_modes=min(5, N_FREQ), include_flat=True)
gsm_maps = sky.basis.deproject(sky.init_coeffs())        # (NPIX, N_FREQ)
print(f"GSM maps: {gsm_maps.shape}  "
      f"range {gsm_maps.min():.0f}–{gsm_maps.max():.0f} K")

# ── 21cm signal model ──────────────────────────────────────────────────────
models_21cm = T21cmModel()
T_21_INJ    = models_21cm(FREQS_HZ, model_index=INJ_MODEL_IDX)   # (N_FREQ,) K
print(f"T_21_INJ (model {INJ_MODEL_IDX}): "
      f"min={T_21_INJ.min()*1e3:.1f} mK  max={T_21_INJ.max()*1e3:.1f} mK")

In [ ]:
# ── Time grid and pointing grid ────────────────────────────────────────────
times = OBS_EPOCH + np.linspace(0, N_DAYS * 86400, N_TIMES, endpoint=False) * u.s

az_grid, alt_grid = np.meshgrid(
    np.deg2rad(AZ_DEG), np.deg2rad(ALT_DEG)
)
azs  = az_grid.ravel()    # (N_ORIENT,)
alts = alt_grid.ravel()   # (N_ORIENT,)

# Precompute beam rotation matrices for all az/alt pointings
rot_ms = beam.get_rotation_matrices(azs, alts)   # (N_ORIENT, 3, 3)

# Galactic pixel unit vectors at NSIDE=8
crds_gal = np.array(healpy.pix2vec(NSIDE, np.arange(NPIX)))   # (3, NPIX)

print(f"Time grid: {times[0].iso} → {times[-1].iso}")
print(f"Az:  {AZ_DEG}")
print(f"Alt: {ALT_DEG}")

In [ ]:
# ── Precompute terrain masks (topocentric, time-dependent) ─────────────────
#
# The terrain is fixed in the topocentric frame.  As Earth rotates, different
# galactic pixels project onto the same terrain mask.  Compute once per time
# step (mask is identical for all az/alt pointings at the same time).

_CACHE = os.path.join(os.path.dirname(os.getcwd()), "notebooks", ".eigsep_cache")
_CACHE = ".eigsep_cache"   # relative to notebooks/
_MASK_FILE = os.path.join(
    _CACHE,
    f"masks_nside{NSIDE}_t{N_TIMES}_az{N_AZ}_alt{N_ALT}.npy"
)
_BVALS_FILE = os.path.join(
    _CACHE,
    f"bvals_nside{NSIDE}_t{N_TIMES}_az{N_AZ}_alt{N_ALT}_f{N_FREQ}.npy"
)

os.makedirs(_CACHE, exist_ok=True)

if os.path.exists(_MASK_FILE) and os.path.exists(_BVALS_FILE):
    print("Loading precomputed masks and beam values from cache …")
    masks_all = np.load(_MASK_FILE)   # (N_TIMES, NPIX) float32
    bvals_all = np.load(_BVALS_FILE)  # (N_TIMES, N_ORIENT, NPIX, N_FREQ) float32
    print(f"  masks: {masks_all.shape}  bvals: {bvals_all.shape}")
else:
    print("Computing terrain masks …", flush=True)
    # Step 1: topocentric pixel coordinates and terrain masks per time step
    crds_top_all = np.zeros((N_TIMES, 3, NPIX), dtype=np.float64)
    masks_all    = np.zeros((N_TIMES, NPIX), dtype=np.float32)
    for ti, t in enumerate(times):
        obs.set_time(t)
        R_g2t            = obs.rot_gal2top()              # (3, 3)
        crds_top         = R_g2t @ crds_gal               # (3, NPIX)
        crds_top_all[ti] = crds_top
        masks_all[ti]    = terrain.mask(crds_top)     # (NPIX,) float32

    print(f"  mean open fraction = {masks_all.mean():.2f}")

    # Step 2: beam values for all (time, pointing) pairs — all frequencies at once
    print("Computing beam values …", flush=True)
    bvals_all = np.zeros((N_TIMES, N_ORIENT, NPIX, N_FREQ), dtype=np.float32)
    for oi in range(N_ORIENT):
        beam.set_az(azs[oi])
        beam.set_alt(alts[oi])
        for ti in range(N_TIMES):
            # beam[crd_top] applies rot_az @ rot_alt to crd_top, then interpolates.
            # Result shape: (NPIX, N_FREQ)
            bvals_all[ti, oi] = np.array(beam[crds_top_all[ti]])
        if (oi + 1) % 8 == 0:
            print(f"  orientation {oi+1}/{N_ORIENT}", flush=True)

    print("Saving to cache …", flush=True)
    np.save(_MASK_FILE,  masks_all)
    np.save(_BVALS_FILE, bvals_all)

print(f"Mean open sky fraction: {masks_all.mean():.2f}")

# ── Pixel visibility fractions ────────────────────────────────────────────
pix_open_frac = masks_all.mean(axis=0)       # (NPIX,) fraction of time each pixel is open
vis_mask      = pix_open_frac > 0            # True = ever-visible (used for eigenmode basis)
print(f"\nPixel visibility:")
print(f"  Never visible (always terrain):  {(pix_open_frac == 0).sum():3d} / {NPIX}")
print(f"  Sometimes visible:               {((pix_open_frac > 0) & (pix_open_frac < 1)).sum():3d} / {NPIX}")
print(f"  Always visible:                  {(pix_open_frac == 1).sum():3d} / {NPIX}")

print(f"Mean beam solid angle at mid-band: "
      f"{bvals_all[..., N_FREQ//2].sum(axis=2).mean() * PIXEL_AREA * (180/np.pi)**2:.1f} deg²")


In [ ]:
# ── Design matrix construction ─────────────────────────────────────────────
#
# Column layout (mirrors BLOOM's build_design_matrix with include_t_rx=False):
#   cols 0..NPIX-1 : beam(p)*mask(p) / bv_sum  — sky pixel contributions
#   col  NPIX      : sum_p beam(p)*(1-mask(p)) / bv_sum — T_gnd fraction
#   col  NPIX+1    : 0.0  — unused (pad for normal_solve interface)
#
# T_rx is NOT included as a column (see run_multifreq for treatment).
# Near-singular modes from rarely-seen pixels are handled by the rcond
# pseudoinverse in normal_solve and eigenvalue-weighted averaging in run_multifreq.

def build_A(fi):
    """
    Build the (N_ROWS, NPIX+2) design matrix for frequency index fi.

    Vectorised over all (time, pointing) pairs.  The actual terrain mask is
    used unmodified; near-zero columns (unobserved pixels) are handled by
    the pseudoinverse cutoff in normal_solve.
    """
    bv = bvals_all[:, :, :, fi].reshape(N_ROWS, NPIX).astype(np.float64)
    m  = np.repeat(masks_all, N_ORIENT, axis=0).astype(np.float64)
    bv_sum = np.sum(bv, axis=1, keepdims=True)
    valid  = bv_sum.ravel() > 0
    A          = np.zeros((N_ROWS, NPIX + 2), dtype=np.float64)
    A[valid, :NPIX] = bv[valid] * m[valid] / bv_sum[valid]
    A[valid,  NPIX] = np.sum(bv[valid] * (1.0 - m[valid]), axis=1) / bv_sum[valid, 0]
    return A


# Quick sanity check at the midband frequency
fi_test  = N_FREQ // 2
A_test   = build_A(fi_test)
bv_flat  = bvals_all[:, :, :, fi_test].reshape(N_ROWS, NPIX)
bv_sum_t = bv_flat.sum(axis=1)
row_sums = A_test[:, :NPIX].sum(axis=1) + A_test[:, NPIX]
valid_t  = bv_sum_t > 0
print(f"Design matrix at {FREQS_MHZ[fi_test]:.1f} MHz: shape {A_test.shape}")
print(f"  Row sum (sky+gnd, valid rows): "
      f"min={row_sums[valid_t].min():.5f}  max={row_sums[valid_t].max():.5f}  (should be 1.0)")

# ── Beam column-norm spectra (for outer-product filter, Cell 7) ───────────
#
# col_norms_all[p, fi] = Σ_rows A[row, p, fi]²
#   = how strongly pixel p is constrained by the data at frequency fi.
# This varies with frequency because the beam is chromatic.
# Its spectral eigenmodes (the 'beam spectral shapes') capture how the
# beam-weighting of each sky direction changes with frequency.

print("Precomputing beam column-norm spectra …", flush=True)
col_norms_all = np.zeros((NPIX, N_FREQ), dtype=np.float64)
for fi in range(N_FREQ):
    bv = bvals_all[:, :, :, fi].reshape(N_ROWS, NPIX).astype(np.float64)
    m  = np.repeat(masks_all, N_ORIENT, axis=0).astype(np.float64)
    bv_sum = np.sum(bv, axis=1, keepdims=True)
    valid  = bv_sum.ravel() > 0
    A_sky = np.zeros((N_ROWS, NPIX))
    A_sky[valid] = bv[valid] * m[valid] / bv_sum[valid]
    col_norms_all[:, fi] = (A_sky ** 2).sum(axis=0)
print(f"  col_norms_all shape: {col_norms_all.shape}")
print(f"  spectral variation (std/mean across freq per pixel): "
      f"{(col_norms_all[vis_mask].std(axis=1) / col_norms_all[vis_mask].mean(axis=1)).mean():.3f}")

# Frequency-averaged pixel weights (constant in f) for monopole extraction.
# FG contribution Σ_p w_const[p]·GSM[p,f]/Σw is a fixed linear combination
# of pixel spectra → always in the span of GSM eigenmodes.
w_const = col_norms_all.mean(axis=1)   # (NPIX,) — never changes with frequency
w_tot   = float(w_const.sum())             # normalisation constant
print(f"  ever-visible pixels with w_const>0: {(w_const>0).sum()} / {NPIX}")


In [ ]:
# ── Foreground eigenmode filter (matched to column-norm weighted monopole) ─
#
# The monopole estimator is Σ_p w_const[p]·T[p] / Σw_const.
# The foreground component of this estimate is Σ_p w_const[p]·GSM[p,f] / Σw_const.
# We build the eigenmode filter from the SAME column-norm weighted GSM so that
# the filter modes exactly span the foreground subspace seen by the estimator.
# Using the unweighted (or vis_mask) GSM causes a spectral mismatch that leaks
# foreground through the filter (~42% FG/sig vs ~5% with matched weighting).
#
# 4 GSM modes is the empirical optimum: the 5th mode starts absorbing signal
# (T_21_filt rms drops from ~5 mK to ~0.9 mK) without reducing FG leakage.

modes = gsm_eigenmodes(w_const[:, np.newaxis] * gsm_maps, N_GSM_MODES)
# modes shape: (N_GSM_MODES+1, N_FREQ)  — N_GSM_MODES GSM modes + 1 flat mode
# The flat mode absorbs T_rx and any DC offset.

N_MODES_TOT = modes.shape[0]
dof         = N_FREQ - N_MODES_TOT

T_21_filt  = eigenmode_filter(T_21_INJ, modes)
T_all      = models_21cm(FREQS_HZ)
T_all_filt = eigenmode_filter(T_all, modes)

T21_filt_rms = float(np.std(T_21_filt))
print(f'Filter: {N_GSM_MODES} GSM modes + 1 flat = {N_MODES_TOT} total  dof={dof}')
print(f'Injected T_21 (model {INJ_MODEL_IDX}): rms after filter = {T21_filt_rms*1e3:.3f} mK')


In [ ]:
# ── Noise model ────────────────────────────────────────────────────────────
#
# Radiometer noise per observation:
#   σ = T_sys / sqrt(Δν · τ)
# where
#   T_sys  = mean(GSM at this freq) + T_rx
#   τ      = total time / N_ROWS  (time budget equally split across all pointings)

tau_per_obs = N_DAYS * 86400.0 / N_ROWS   # seconds per (time, pointing) slot
sigma_noise = np.array([
    (float(gsm_maps[:, fi].mean()) + T_RX_K) / np.sqrt(DELTA_NU_HZ * tau_per_obs)
    for fi in range(N_FREQ)
])   # (N_FREQ,)  [K]

print(f"τ per obs:  {tau_per_obs:.1f} s")
print(f"σ_noise:    {sigma_noise.min()*1e3:.1f}–{sigma_noise.max()*1e3:.1f} mK  "
      f"(at {FREQS_MHZ[np.argmin(sigma_noise)]:.0f}–"
      f"{FREQS_MHZ[np.argmax(sigma_noise)]:.0f} MHz)")

In [ ]:
# ── Core multi-frequency inversion ────────────────────────────────────────

def run_multifreq(noise_scale=1.0, noise_seed=0):
    """
    Per-frequency sky inversion for EIGSEP.

    For each frequency:
      1. Build design matrix A  (NPIX+2 columns: sky | T_gnd | 0)
      2. Simulate y = A @ [sky + T_21 | T_gnd | 0] + T_RX_K + noise
      3. Solve A @ x = y via pseudoinverse (rcond zeroes singular modes)
      4. Sky monopole via frequency-averaged information-content weights:
         w_const[p] = mean_f(col_norms_sq[p,f])  —  constant in frequency.
         Because the weights are frequency-independent, the FG component
         Σ_p w_const[p]·GSM[p,f] is a fixed linear combination of pixel
         spectra, guaranteed to lie in the GSM eigenmode span.  Poorly-
         observed pixels (small average col_norm) are down-weighted without
         introducing frequency-varying bias.
      5. Propagate noise: σ_mono = σ * sqrt(e^T (A^T A)^{-1} e)
         where e = w_const / sum(w_const), also constant in frequency.

    Parameters
    ----------
    noise_scale : float
    noise_seed  : int

    Returns
    -------
    T_sky_mean_est : (N_FREQ,)
    SIGMA_MONO     : (N_FREQ,)
    """
    T_sky_mean_est = np.empty(N_FREQ)
    SIGMA_MONO     = np.empty(N_FREQ)
    rng = np.random.default_rng(noise_seed)

    # Frequency-independent estimator vector (same e_sky for every channel)
    w_tot = float(w_const.sum())
    e_sky = np.zeros(NPIX + 2)
    if w_tot > 0:
        e_sky[:NPIX] = w_const / w_tot

    for fi in range(N_FREQ):
        gsm_f   = gsm_maps[:, fi].astype(np.float64)
        sky_f   = gsm_f + T_21_INJ[fi]
        sigma_f = sigma_noise[fi] * noise_scale

        A_f    = build_A(fi)
        x_true = np.concatenate([sky_f, [T_GND_K], [0.0]])
        y_f    = A_f @ x_true + T_RX_K + rng.normal(scale=sigma_f, size=N_ROWS)

        res_f  = normal_solve(A_f, y_f, NPIX)

        # Frequency-averaged weights: FG component is a fixed pixel-spectrum
        # linear combination → lies in GSM eigenmode span → outer-product
        # filter removes it exactly.
        # nansum handles NaN sky_map at unobserved pixels (w_const already 0 there).
        if w_tot > 0:
            T_sky_mean_est[fi] = float(np.nansum(res_f['sky_map'] * w_const) / w_tot)
        else:
            T_sky_mean_est[fi] = np.nan

        # Noise propagation with the same constant e_sky
        Ve = res_f['eigenvectors'].T @ e_sky
        SIGMA_MONO[fi] = sigma_f * np.sqrt(
            float(np.dot(Ve ** 2, res_f['inv_eigenvalues']))
        )

    return T_sky_mean_est, SIGMA_MONO



In [ ]:
_, S, Vt = np.linalg.svd(gsm_maps[vis_mask], full_matrices=False)
plt.figure()
plt.semilogy(S, '.')

In [ ]:
_, S_beam, Vt_beam = np.linalg.svd(col_norms_all[vis_mask], full_matrices=False)
plt.figure()
plt.semilogy(S_beam, '.')

In [ ]:
prod = [1e5 * np.ones(S.size)]
for i in range(S.size):
    for j in range(S.size):
        prod.append(S[i] * Vt[i] * S_beam[j] * Vt_beam[j])
prod = np.array(prod)

In [ ]:
_, S_tot, Vt_tot = np.linalg.svd(prod, full_matrices=False)
plt.figure()
plt.semilogy(S_tot, '.')
modes_explore = Vt_tot[:10]

In [ ]:
# ── Noiseless run — foreground leakage check ──────────────────────────────

print("Noiseless run …", flush=True)
T_est_nl, _ = run_multifreq(noise_scale=0.0)

resid_nl   = eigenmode_filter(T_est_nl, modes)
FG_leakage = resid_nl - T_21_filt

print(f"  FG leakage rms  = {np.std(FG_leakage)*1e3:.4f} mK")
print(f"  T_21_filt rms   = {T21_filt_rms*1e3:.4f} mK")
print(f"  FG/signal ratio = {np.std(FG_leakage)/T21_filt_rms:.4f}")
print(f"  Max |noiseless recovery error| in sky pixels:", flush=True)

# Also report per-frequency noiseless sky monopole vs truth
truth_mono = np.array([float(gsm_maps[:, fi].mean()) + T_21_INJ[fi] for fi in range(N_FREQ)])
mono_err   = T_est_nl - truth_mono
print(f"  Δ(sky monopole) noiseless: rms={np.std(mono_err)*1e3:.3f} mK  "
      f"max={np.abs(mono_err).max()*1e3:.3f} mK")

In [ ]:
plt.figure()
plt.plot(T_est_nl)
plt.plot(resid_nl)

In [ ]:
# ── Noisy run (baseline noise level) ──────────────────────────────────────

print("Noisy run (noise_scale=1) …", flush=True)
T_est, SIGMA_MONO = run_multifreq(noise_scale=1.0, noise_seed=42)

resid_est  = eigenmode_filter(T_est, modes)
noise_term = eigenmode_filter(T_est - T_est_nl, modes)

sigma_mean = float(np.mean(SIGMA_MONO))
chi2_wt    = np.sum(((resid_est - T_all_filt) / SIGMA_MONO[np.newaxis, :])**2,
                    axis=1) / dof
chi2_noise = float(np.sum((noise_term / SIGMA_MONO)**2) / dof)   # should be ~1 if calibrated

print(f"  SIGMA_MONO range: {SIGMA_MONO.min()*1e3:.2f}–{SIGMA_MONO.max()*1e3:.2f} mK")
print(f"  sigma_mean:       {sigma_mean*1e3:.3f} mK")
print(f"  resid rms:        {np.std(resid_est)*1e3:.4f} mK")
print(f"  noise term rms:   {np.std(noise_term)*1e3:.4f} mK")
print(f"  FG leakage rms:   {np.std(FG_leakage)*1e3:.4f} mK  "
      f"({np.std(FG_leakage)/sigma_mean:.2f}× sigma_mean)")
print(f"  chi²/dof noise:   {chi2_noise:.4f}  (expected ~1 if noise calibrated)")
print(f"  chi²/dof model {INJ_MODEL_IDX}: {chi2_wt[INJ_MODEL_IDX]:.3f}")
print(f"  Best model:       {np.argmin(chi2_wt)} (chi²/dof={chi2_wt.min():.3f})")

In [ ]:
# ── Scale noise to target SNR ──────────────────────────────────────────────
#
# Follow the same procedure as bloom21cm/test_multifreq.py Step 2:
# choose sigma_scale so that noise rms ≈ T_21_filt_rms / TARGET_SNR.

TARGET_SNR   = 2.0
noise_rms_1  = float(np.std(noise_term))                    # noise rms at scale=1
target_noise = T21_filt_rms / TARGET_SNR
sigma_scale  = target_noise / noise_rms_1 if noise_rms_1 > 0 else 1.0

print(f"Noise rms at scale=1:  {noise_rms_1*1e3:.4f} mK")
print(f"T_21_filt rms:         {T21_filt_rms*1e3:.4f} mK")
print(f"Target noise rms:      {target_noise*1e3:.4f} mK  (÷{TARGET_SNR:.0f})")
print(f"=> sigma_scale:        {sigma_scale:.4f}")

if noise_rms_1 * sigma_scale > np.std(FG_leakage):
    print(f"Noise ({noise_rms_1*sigma_scale*1e3:.4f} mK) > FG leakage "
          f"({np.std(FG_leakage)*1e3:.4f} mK) ✓")
else:
    print(f"WARNING: noise < FG leakage — sigma_scale may be too small")

In [ ]:
# ── Signal recovery run at target SNR ─────────────────────────────────────

print(f"Signal recovery run (sigma_scale={sigma_scale:.4f}) …", flush=True)
T_est_s, SIGMA_MONO_s = run_multifreq(noise_scale=sigma_scale, noise_seed=100)

resid_est_s  = eigenmode_filter(T_est_s, modes)
noise_term_s = eigenmode_filter(T_est_s - T_est_nl, modes)

chi2_thresh  = 1.0 + 2.0 * np.sqrt(2.0 / dof)
chi2_wt_s    = np.sum(((resid_est_s - T_all_filt) / SIGMA_MONO_s[np.newaxis, :])**2,
                       axis=1) / dof
chi2_noise_s = float(np.sum((noise_term_s / SIGMA_MONO_s)**2) / dof)
chi2_model0  = float(chi2_wt_s[INJ_MODEL_IDX])
rank_model0  = int(np.sum(chi2_wt_s <= chi2_model0))
n_models     = chi2_wt_s.shape[0]
pct_rank     = rank_model0 / n_models * 100.0

# Per-frequency SNR
SNR_s          = T_21_filt / SIGMA_MONO_s
SNR_combined_s = float(np.sqrt(np.sum(SNR_s**2)))

pass_noise = 0.3 < chi2_noise_s < 3.0
pass_chi2  = chi2_model0 < chi2_thresh
pass_rank  = pct_rank <= 20.0
overall    = pass_noise and pass_chi2 and pass_rank

print(f"\n  dof                    = {dof}")
print(f"  SIGMA_MONO range       = {SIGMA_MONO_s.min()*1e3:.2f}–{SIGMA_MONO_s.max()*1e3:.2f} mK")
print(f"  noise rms (filtered)   = {np.std(noise_term_s)*1e3:.4f} mK")
print(f"  FG leakage rms         = {np.std(FG_leakage)*1e3:.4f} mK")
print(f"  T_21_filt rms          = {T21_filt_rms*1e3:.4f} mK")
print(f"  chi²/dof (noise only)  = {chi2_noise_s:.4f}  (expected ~1)")
print(f"  chi²/dof model 0       = {chi2_model0:.4f}  (threshold {chi2_thresh:.4f})")
print(f"  Best model             = {np.argmin(chi2_wt_s)}  "
      f"(chi²/dof={chi2_wt_s.min():.4f})")
print(f"  Rank model 0           = {rank_model0}/{n_models}  ({pct_rank:.1f}th pct)")
print(f"  Combined SNR           = {SNR_combined_s:.2f}")
print()
print(f"  {'PASS' if pass_noise else 'FAIL'}: noise chi²/dof = {chi2_noise_s:.3f}  [0.3–3.0]")
print(f"  {'PASS' if pass_chi2  else 'FAIL'}: model-0 chi²/dof = {chi2_model0:.3f} < {chi2_thresh:.3f}")
print(f"  {'PASS' if pass_rank  else 'FAIL'}: rank = {rank_model0}/{n_models}  (≤20th pct)")
print(f"\n  OVERALL: {'PASS — signal recovery confirmed' if overall else 'FAIL — check N_EIG_MODES / observing params'}")

In [ ]:
# ── SNR per frequency ──────────────────────────────────────────────────────

print(f"  {'Freq [MHz]':>10}  {'T_21_filt [mK]':>15}  {'σ [mK]':>8}  {'SNR':>7}")
print("  " + "-" * 46)
for f, s21, sig, snr in zip(FREQS_MHZ, T_21_filt * 1e3, SIGMA_MONO_s * 1e3, SNR_s):
    print(f"  {f:10.1f}  {s21:15.3f}  {sig:8.3f}  {snr:7.3f}")
print(f"\n  Combined SNR = sqrt(Σ SNR²) = {SNR_combined_s:.2f}")
print(f"  sigma_scale = {sigma_scale:.4f}  (scales total obs time by {sigma_scale**2:.2e}×)")

In [ ]:
# ── Terrain map visualization ──────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Show terrain mask averaged over all times (fraction of time each pixel is visible)
vis_fraction = masks_all.mean(axis=0)   # (NPIX,) — time-averaged visibility
healpy.mollview(vis_fraction, fig=fig, sub=(1, 2, 1),
                title="Mean sky visibility (terrain mask, galactic frame)",
                unit="fraction", coord=['G'])

# GSM at midband
healpy.mollview(gsm_maps[:, N_FREQ // 2] / 1e3, fig=fig, sub=(1, 2, 2),
                title=f"GSM at {FREQS_MHZ[N_FREQ//2]:.0f} MHz (NSIDE=8)",
                unit="kK", coord=['G'])
plt.tight_layout()
plt.savefig(os.path.join(_CACHE, "terrain_sky_map.png"), dpi=100)
plt.show()
print(f"Visibility fraction: min={vis_fraction.min():.2f}  max={vis_fraction.max():.2f}  "
      f"mean={vis_fraction.mean():.2f}")

In [ ]:
# ── Diagnostic plots ───────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Panel 1 — residual spectrum
ax = axes[0]
ax.fill_between(FREQS_MHZ, -SIGMA_MONO_s * 1e3, SIGMA_MONO_s * 1e3,
                alpha=0.25, color='C0', label='±1σ per-freq')
ax.plot(FREQS_MHZ, resid_est_s * 1e3,   'k.-', ms=5, label='residual (noisy)')
ax.plot(FREQS_MHZ, T_21_filt * 1e3,     'r--', lw=2, label='T_21_filt (injected)')
ax.plot(FREQS_MHZ, FG_leakage * 1e3,    'g-',  alpha=0.7, label='FG leakage')
ax.set_xlabel('Frequency [MHz]')
ax.set_ylabel('ΔT [mK]')
ax.set_title(f'Residual  (model-0 chi²/dof={chi2_model0:.2f})')
ax.legend(fontsize=8)

# Panel 2 — SIGMA_MONO per frequency
ax = axes[1]
ax.semilogy(FREQS_MHZ, SIGMA_MONO_s * 1e3, 'b-o', ms=4, label='SIGMA_MONO (scaled)')
ax.semilogy(FREQS_MHZ, sigma_noise * 1e3,  'r--', lw=1, label='σ_radiometer')
ax.set_xlabel('Frequency [MHz]')
ax.set_ylabel('σ [mK]')
ax.set_title('Per-frequency noise')
ax.legend(fontsize=9)

# Panel 3 — chi²/dof CDF over model library
ax = axes[2]
sorted_chi2 = np.sort(chi2_wt_s)
ax.plot(sorted_chi2, np.arange(1, n_models + 1) / n_models * 100, 'k-', lw=1.5)
ax.axvline(chi2_model0, color='r', ls='--', lw=2,
           label=f'Model 0  χ²/dof={chi2_model0:.2f}')
ax.axvline(1.0, color='g', ls=':', lw=1.5, label='χ²/dof=1')
ax.set_xlabel('χ²/dof')
ax.set_ylabel('Cumulative % of models')
ax.set_title('CDF of model χ²/dof')
ax.legend(fontsize=8)

plt.suptitle(
    f'EIGSEP NSIDE=8 Recovery  |  N_GSM_MODES={N_GSM_MODES}  '
    f'sigma_scale={sigma_scale:.3f}  SNR_combined={SNR_combined_s:.1f}',
    y=1.02, fontsize=11
)
plt.tight_layout()
plt.savefig(os.path.join(_CACHE, "eigsep_recovery_v000.png"), dpi=100, bbox_inches='tight')
plt.show()
print("Saved plots.")


In [ ]:
# ── N_GSM_MODES scan ──────────────────────────────────────────────────────
#
# Scan over the number of GSM eigenmodes in the matched filter.
# 4 modes is the empirical optimum (5th mode starts absorbing signal).
# T_est_scan is independent of modes; precompute it once.

scan_gsm = [2, 3, 4, 5, 6]

print('Precomputing noiseless inversions …', flush=True)
T_est_scan = np.empty(N_FREQ)
for fi_s in range(N_FREQ):
    bv_s  = bvals_all[:, :, :, fi_s].reshape(N_ROWS, NPIX).astype(np.float64)
    m_s   = np.repeat(masks_all, N_ORIENT, axis=0).astype(np.float64)
    bvs_s = bv_s.sum(axis=1, keepdims=True)
    vld_s = bvs_s.ravel() > 0
    A_s   = np.zeros((N_ROWS, NPIX + 2))
    A_s[vld_s, :NPIX] = bv_s[vld_s] * m_s[vld_s] / bvs_s[vld_s]
    A_s[vld_s,  NPIX] = np.sum(bv_s[vld_s]*(1-m_s[vld_s]), axis=1) / bvs_s[vld_s, 0]
    x_s  = np.concatenate([gsm_maps[:, fi_s]+T_21_INJ[fi_s], [T_GND_K], [0.0]])
    r_s  = normal_solve(A_s, A_s @ x_s + T_RX_K, NPIX)
    T_est_scan[fi_s] = float(np.nansum(r_s['sky_map']*w_const)/w_tot) if w_tot > 0 else np.nan
print('  done.')

print(f"{'N_GSM':>6}  {'N_modes':>7}  {'dof':>4}  {'T_21_filt rms [mK]':>20}  {'FG/sig':>8}")
print('-' * 55)
for n_gsm in scan_gsm:
    m_s      = gsm_eigenmodes(w_const[:, np.newaxis] * gsm_maps, n_gsm)
    T21_f_s  = eigenmode_filter(T_21_INJ, m_s)
    T21_rms  = float(np.std(T21_f_s))
    FG_s     = eigenmode_filter(T_est_scan, m_s) - T21_f_s
    ratio    = float(np.std(FG_s) / T21_rms) if T21_rms > 1e-10 else np.inf
    print(f'  {n_gsm:4d}  {m_s.shape[0]:7d}  {N_FREQ-m_s.shape[0]:4d}  '
          f'{T21_rms*1e3:20.3f}  {ratio:8.4f}')


In [ ]:
resid_est